# 03 - Data Preprocessing

Goal:
Prepare the 2025 SHED dataset for modeling household financial fragility.

This notebook will:
- define the binary fragility target
- remove identifiers, weights, imputation flags, and target-leakage variables
- inspect and handle structural missingness
- separate categorical and numeric predictors
- create a reproducible preprocessing pipeline
- create the train/test split

In [2]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
DATA_PATH = os.path.join(
    os.getcwd(),
    "..",
    "data",
    "raw",
    "SHED 2025 CSV Raw.csv"
)

df = pd.read_csv(DATA_PATH, low_memory=False)

In [4]:
print(df.shape)
print(df["shedid"].nunique())

(12934, 815)
12934


In [5]:
target = df["pay_casheqv"].map({
    "Yes": 0,
    "No": 1
})

print(target.value_counts(dropna=False))

pay_casheqv
0    8398
1    4536
Name: count, dtype: int64


In [6]:
assert target.isna().sum() == 0

## Target Definition

`pay_casheqv` identifies whether the respondent could cover a $400
emergency expense using cash or its equivalent.

For modeling:

- 0 = financially resilient on this measure
- 1 = financially fragile on this measure

In [7]:
sample_weight = df["weight"].copy()
population_weight = df["weight_pop"].copy()

In [ ]:
id_columns = [
    "shedid"
]

weight_columns = [
    "weight",
    "weight_pop",
    "panel_weight",
    "panel_weight_pop"
]

target_columns = [
    "pay_casheqv"
]

target_leakage_columns = [
    "EF3_a",
    "EF3_b",
    "EF3_c",
    "EF3_d",
    "EF3_e",
    "EF3_f",
    "EF3_g",
    "EF3_h"
]

In [12]:
iflag_columns = [
    col for col in df.columns
    if col.endswith("_iflag")
]

len(iflag_columns)

357

In [14]:
excluded_columns = (
    id_columns
    + weight_columns
    + target_columns
    + target_leakage_columns
    + iflag_columns
)

X = df.drop(
    columns=excluded_columns,
    errors="ignore"
).copy()

print("Original columns:", df.shape[1])
print("Candidate predictors:", X.shape[1])

Original columns: 815
Candidate predictors: 444


In [15]:
missing_report = pd.DataFrame({
    "missing_n": X.isna().sum(),
    "missing_pct": X.isna().mean() * 100
})

missing_report = missing_report.sort_values(
    "missing_pct",
    ascending=False
)

missing_report.head(30)

,missing_n,missing_pct
BK49B,12823,99.141797
A8_d,12818,99.103139
A8_c,12812,99.056750
A8_e,12808,99.025823
S18,12794,98.917582
R5C_c,12773,98.755219
R5C_d,12773,98.755219
R5C_b,12773,98.755219
R5C_a,12773,98.755219
S21,12655,97.842895


In [17]:
missing_report.to_csv(
    "../outputs/tables/missingness_report.csv"
)

In [18]:
categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical:", len(categorical_features))
print("Numeric:", len(numeric_features))

Categorical: 417
Numeric: 27


In [22]:
categorical_features

['xlaptop',
 'L0_a',
 'L0_b',
 'L0_c',
 'L0_d',
 'L0_e',
 'L0B_a',
 'L0B_b',
 'L0B_c',
 'L0C',
 'L0E',
 'B2',
 'B3',
 'B3A_a',
 'B3A_b',
 'B3A_c',
 'B3A_d',
 'B3A_e',
 'B3A_f',
 'B3B_a',
 'B3B_b',
 'B3B_c',
 'B3B_d',
 'B3B_e',
 'B3B_f',
 'B7_a',
 'B7_b',
 'X11_none',
 'X12_a',
 'X12_b',
 'X12_c',
 'X12_d',
 'X12_e',
 'X12_f',
 'X12_g',
 'X13',
 'CG0',
 'CG1A',
 'CG7_a',
 'CG7_b',
 'CG7_c',
 'CG7_d',
 'CG7_e',
 'CG7_f',
 'CG7_g',
 'CG8',
 'CG3',
 'CG4',
 'CG5_a',
 'CG5_b',
 'CG5_c',
 'CG5_d',
 'CG5_e',
 'CG5_f',
 'CG6',
 'D1A',
 'D4',
 'D48',
 'D3B',
 'D1E',
 'D22_a',
 'D22_b',
 'D22_c',
 'D22_d',
 'D22_e',
 'D22_f',
 'D22_g',
 'D22_h',
 'D22_i',
 'D3A',
 'D3E',
 'D28_a',
 'D28_b',
 'D50',
 'D51',
 'D52_a',
 'D52_b',
 'D52_c',
 'D52_d',
 'D52_e',
 'D52_f',
 'D30',
 'D34A',
 'D34D',
 'D36B',
 'D44_f',
 'D44_a',
 'D44_b',
 'D44_c',
 'D44_d',
 'D44_e',
 'D37A',
 'D38_a',
 'D38_b',
 'D38_c',
 'D38_d',
 'D38_f',
 'D39',
 'D1B',
 'D5',
 'D49',
 'GH1',
 'GH12',
 'GH12A',
 'GH14',
 'GH14A',
 'G

In [23]:
numeric_features

['duration',
 'CG2',
 'R3',
 'M4',
 'BK49A',
 'BK49B',
 'dov_ef7',
 'DOV_X11',
 'ppage',
 'pphhsize',
 'ppkid017',
 'ppt18ov',
 'ppsi2302',
 'ppcmdate',
 'ppp1date',
 'ppp2date',
 'pphidate',
 'pph1date',
 'ppfsdate',
 'ppmpdate',
 'ppcpdate',
 'ppsidate',
 'ppc2date',
 'field_month',
 'field_day',
 'pphhsize5',
 'year']

In [24]:
categorical_numeric = [
    "dov_ef7", "DOV_X11"
]

categorical_features += categorical_numeric

numeric_features = [
    col for col in numeric_features
    if col not in categorical_numeric
]

In [25]:
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="NOT_ASKED"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "scaler",
        StandardScaler()
    )
])

In [26]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

In [28]:
X_train, X_test, y_train, y_test, weight_train, weight_test = (
    train_test_split(
        X,
        target,
        sample_weight,
        test_size=0.20,
        random_state=42,
        stratify=target
    )
)

print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(10347, 444)
(2587, 444)
pay_casheqv
0    0.64927
1    0.35073
Name: proportion, dtype: float64
pay_casheqv
0    0.649401
1    0.350599
Name: proportion, dtype: float64


In [29]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print(
    "Training shape before:",
    X_train.shape
)

print(
    "Training shape after:",
    X_train_prepared.shape
)

Training shape before: (10347, 444)
Training shape after: (10347, 2403)


In [30]:
feature_names = preprocessor.get_feature_names_out()

print(len(feature_names))
print(feature_names[:50])

2403
['numeric__duration' 'numeric__CG2' 'numeric__R3' 'numeric__M4'
 'numeric__BK49A' 'numeric__BK49B' 'numeric__ppage' 'numeric__pphhsize'
 'numeric__ppkid017' 'numeric__ppt18ov' 'numeric__ppsi2302'
 'numeric__ppcmdate' 'numeric__ppp1date' 'numeric__ppp2date'
 'numeric__pphidate' 'numeric__pph1date' 'numeric__ppfsdate'
 'numeric__ppmpdate' 'numeric__ppcpdate' 'numeric__ppsidate'
 'numeric__ppc2date' 'numeric__field_month' 'numeric__field_day'
 'numeric__pphhsize5' 'numeric__year' 'numeric__missingindicator_CG2'
 'numeric__missingindicator_R3' 'numeric__missingindicator_M4'
 'numeric__missingindicator_BK49A' 'numeric__missingindicator_BK49B'
 'numeric__missingindicator_ppsi2302' 'numeric__missingindicator_ppp1date'
 'numeric__missingindicator_ppp2date' 'numeric__missingindicator_pphidate'
 'numeric__missingindicator_pph1date' 'numeric__missingindicator_ppfsdate'
 'numeric__missingindicator_ppmpdate' 'numeric__missingindicator_ppcpdate'
 'numeric__missingindicator_ppsidate' 'numeric__m

In [31]:
train_indices = X_train.index
test_indices = X_test.index

In [32]:
split_info = pd.DataFrame({
    "index": df.index,
    "split": "unused"
})

split_info.loc[train_indices, "split"] = "train"
split_info.loc[test_indices, "split"] = "test"

In [33]:
split_info.to_csv(
    "../outputs/tables/train_test_split.csv",
    index=False
)

In [34]:
assert len(X_train) + len(X_test) == len(df)
assert len(y_train) + len(y_test) == len(target)

assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

assert "shedid" not in X.columns
assert "pay_casheqv" not in X.columns

for col in target_leakage_columns:
    assert col not in X.columns

## Preprocessing Summary

- Target: binary financial fragility based on `pay_casheqv`
- Positive class: unable to cover a $400 emergency using cash/equivalent
- Direct target variables (`EF3_a` through `EF3_h`) removed
- Respondent IDs removed
- Survey weights retained separately
- `_iflag` columns removed from predictors
- Structural categorical missingness preserved as `NOT_ASKED`
- Numeric missingness median-imputed with missingness indicators
- Categorical variables one-hot encoded
- Numeric variables standardized
- Train/test split: 80/20
- Split stratified on the fragility target
- Preprocessing fit only on training data